# Shared Blob Workspace — a project drive for a crew of agents

This notebook builds **two prompt agents** on the Foundry managed harness that collaborate
through one **shared, durable workspace** backed by Azure Blob Storage. Each agent mounts the
same blob container as a plain folder (the *project drive*), so **what one agent produces, the
other builds on** — even in a different chat, on a different day.

- **📚 Scout** *(research)* — distills raw source material into structured findings.
- **✍️ Quill** *(analyst/writer)* — turns those findings into a polished report.

They work like **teammates sharing a drive**: each picks up what the other produced and keeps
the project moving. The only custom tool either one needs is the **`workspace-mount`** skill,
which mounts the container as a folder using the *agent's own managed identity* (MSI) — no key
or SAS ever enters the model's context.

## Why a blob mount (and what it uniquely gives us)

A Foundry **conversation** runs on one warm *Hand* (sandbox): files written in one turn persist
to later turns of the **same** conversation. But a **new** conversation gets a **fresh Hand** —
its local disk is empty. The **blob mount is what survives that boundary**: mount the same
container from any conversation or any agent and the files are all still there. That is exactly
what lets Scout (Session 1) hand off to Quill (Session 2, new chat, different agent) with nothing
re-explained.

> This uses `blobfuse2` inside the Hand — an **interim** pattern for giving hosted agents a
> durable, shareable workspace until native blob-volume mounting is available.

## The walkthrough — 5 prompts, 2 agents, 3 sessions

| Session (conversation) | Agent | Prompts |
|---|---|---|
| 1 | Scout | research the 5 companies → deepen with sentiment + funding |
| 2 *(new chat)* | Quill | draft the report → refine (5-bullet summary + pricing/features quadrant) |
| 3 *(new chats)* | Scout, then Quill | a new competitor launches → Scout profiles it → Quill folds it in |

Each session is a **separate conversation = separate Hand**; the work flows between them only
through the mounted blob. Watch the `x-agent-session-id` change between sessions while the data
still carries over — that is the whole point.

## Prerequisites

- An Azure AI Foundry **project** with a deployed model. Need one? Deploy
  [`infrastructure-setup-bicep/40-basic-agent-setup`](../../../../infrastructure/infrastructure-setup-bicep/40-basic-agent-setup).
- **Azure CLI** logged in (`az login`). Your dev identity is used only to *author* (publish the
  skill, create the toolbox + connection, create the agents, assign roles) — it needs
  **Azure AI User** on the project and rights to create role assignments on the storage account.
- A **storage account** (same tenant) and a **blob container** to use as the project drive.
- Python 3.10+ and a package manager (`pip` + `venv`, or `uv`).

### Author-time setup

Copy `.env.sample` to `.env` and fill in the values. Then **seed the project drive** once with
the sample's starter material (see [`data/README.md`](./data/README.md)):

```bash
# from this folder, with AZURE_STORAGE_ACCOUNT / WORKSPACE_CONTAINER exported
az storage blob upload-batch \
  --account-name "$AZURE_STORAGE_ACCOUNT" \
  --destination  "$WORKSPACE_CONTAINER/raw-inbox" \
  --source ./data --auth-mode login
```

> **Order matters:** each agent's managed identity does not exist until the agent is *created*,
> so its **Storage Blob Data Contributor** grant happens **after** create (Step 3), not before.

In [1]:
import os
import dotenv

from sample_config import (
    QUILL_AGENT_NAME,
    SCOUT_AGENT_NAME,
    TOOLBOX_CONNECTION_NAME,
    toolbox_mcp_url,
)

dotenv.load_dotenv()

PROJECT_ENDPOINT = os.environ["AZURE_AI_PROJECT_ENDPOINT"].rstrip("/")
MODEL = os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"]
PROJECT_RESOURCE_ID = os.environ["PROJECT_RESOURCE_ID"]
STORAGE_ACCOUNT = os.environ["AZURE_STORAGE_ACCOUNT"]
WORKSPACE_CONTAINER = os.environ["WORKSPACE_CONTAINER"]

# The toolbox + connection are created by provision_skills.py (Step 1). The MCP tool points at
# the toolbox's data-plane MCP endpoint; the connection (referenced by name) supplies
# AgenticIdentityToken auth so each agent's OWN identity authorizes the call.
TOOLBOX_MCP_URL = f"{toolbox_mcp_url(PROJECT_ENDPOINT)}?api-version=v1"

print("Project:   ", PROJECT_ENDPOINT)
print("Model:     ", MODEL)
print("Toolbox:   ", TOOLBOX_MCP_URL)
print("Connection:", TOOLBOX_CONNECTION_NAME)
print("Drive:     ", f"{STORAGE_ACCOUNT}/{WORKSPACE_CONTAINER}")
print("Agents:    ", SCOUT_AGENT_NAME, "+", QUILL_AGENT_NAME)

Project:    https://achauhan-ai-mha-ws2.services.ai.azure.com/api/projects/achauhan-ai-mha-ws2-prj
Model:      gpt-5.4
Toolbox:    https://achauhan-ai-mha-ws2.services.ai.azure.com/api/projects/achauhan-ai-mha-ws2-prj/toolboxes/workspace-tools/mcp?api-version=v1
Connection: workspace-tools-toolbox
Drive:      auditdatademo/sdk-repo
Agents:     scout-research + quill-analyst


## Step 1 — Publish the skill, toolbox, and connection (run once)

`provision_skills.py` packages every file under `skills/workspace-mount/` (including
`scripts/mount_workspace.sh`), publishes it as a **skill**, **creates the toolbox**
(`workspace-tools`) with that skill as its default version, and **creates the agent-identity
project connection** (`workspace-tools-toolbox`) that fronts the toolbox's MCP endpoint. Both
agents share this one toolbox. Re-run whenever you edit the skill.

In [2]:
import subprocess, sys

result = subprocess.run([sys.executable, "provision_skills.py"], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit("provision_skills.py failed — fix the error above before continuing.")

Publishing 1 skill(s) to https://achauhan-ai-mha-ws2.services.ai.azure.com/api/projects/achauhan-ai-mha-ws2-prj ...
  published skill 'workspace-mount' (version 1)
Attaching to toolbox 'workspace-tools' ...
  created toolbox and attached 1 skill(s) to toolbox 'workspace-tools' (version 1, now default)
Creating the toolbox connection (AgenticIdentityToken) ...
  connection 'workspace-tools-toolbox' -> https://achauhan-ai-mha-ws2.services.ai.azure.com/api/projects/achauhan-ai-mha-ws2-prj/toolboxes/workspace-tools/mcp
Done. Both agents reference the toolbox via this connection (no env var needed):
  /subscriptions/2d385bf4-0756-4a76-aa95-28bf9ed3b625/resourceGroups/achauhan-mha-ws2/providers/Microsoft.CognitiveServices/accounts/achauhan-ai-mha-ws2/projects/achauhan-ai-mha-ws2-prj/connections/workspace-tools-toolbox



## Step 2 — Create the two prompt agents

Each agent gets a single tool — **`MCPTool`** referencing the shared toolbox connection. We do
**not** add `CodeInterpreterTool`: on the managed harness the code interpreter is a *separate*
sandbox that would not see the blob mount. The agent instead reads/writes files with the Hand's
own shell/Python **in the same sandbox where the `workspace-mount` skill mounted the drive**.

The instructions come from `agents/<name>/instructions.md`; we append a small **"Your project
drive"** block with this deployment's storage account + container so the agent knows what to
mount. Creating each agent **materializes its managed identity**, whose principal id we grant
roles to in Step 3.

In [2]:
from pathlib import Path

from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import MCPTool, PromptAgentDefinition
from azure.identity import DefaultAzureCredential

project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=DefaultAzureCredential())

DRIVE_BLOCK = f'''

## Your project drive (this deployment)

- Storage account: `{STORAGE_ACCOUNT}`
- Container: `{WORKSPACE_CONTAINER}`
- Mount path: `/workspace/project`

At the start of every session, invoke the **workspace-mount** skill's `mount` command, passing
this account and container, e.g.:

    mount --account {STORAGE_ACCOUNT} --container {WORKSPACE_CONTAINER}

Read `/workspace/project/ai-notes-analysis/status.md` first. Use the Hand's own shell/Python
(the same sandbox) for all file reads/writes — do not use a separate code-interpreter tool.
Before you finish a turn that wrote anything, invoke the workspace-mount skill's `flush` command
so the next session/agent sees your work.
'''

def load_instructions(agent_dir: str) -> str:
    base = Path("agents") / agent_dir / "instructions.md"
    return base.read_text(encoding="utf-8") + DRIVE_BLOCK

def create_agent(agent_name: str, agent_dir: str):
    definition = PromptAgentDefinition(
        model=MODEL,
        instructions=load_instructions(agent_dir),
        temperature=0,
        tools=[
            MCPTool(
                server_url=TOOLBOX_MCP_URL,
                server_label="toolbox",
                require_approval="never",
                project_connection_id=TOOLBOX_CONNECTION_NAME,
            ),
        ],
    )
    # Run in the managed harness so the skill executes server-side under the agent identity.
    definition["harness"] = "ghcp"
    agent = project_client.agents.create_version(agent_name=agent_name, definition=definition)
    print(f"Created {agent.name} (id={agent.id}, version={agent.version})")
    return agent

scout = create_agent(SCOUT_AGENT_NAME, "scout")
quill = create_agent(QUILL_AGENT_NAME, "quill")

Created scout-research (id=scout-research:1, version=1)
Created quill-analyst (id=quill-analyst:1, version=1)


## Step 3 — Grant each agent identity its runtime role

Both agents now have Entra service principals. Each needs **Storage Blob Data Contributor** on
the storage account (to mount the container read-write via MSI). The toolbox call itself is
authorized by the connection's `AgenticIdentityToken` (the agent's `Azure AI User` on the
project — typically already present from project setup).

Run the printed `az` commands (or set `APPLY = True` to run them here), then wait ~5–10 minutes
for the data-plane RBAC to propagate before Step 4.

In [3]:
import subprocess, os

ROLE_STORAGE_BLOB_DATA_CONTRIBUTOR = "Storage Blob Data Contributor"
STORAGE_RESOURCE_ID = os.environ.get("STORAGE_RESOURCE_ID", "<storage-account-resource-id>")

APPLY = False  # set True to actually create the role assignments (needs STORAGE_RESOURCE_ID)

def principal_id(agent_name: str) -> str:
    out = subprocess.run(
        ["az", "rest", "--method", "GET",
         "--url", f"{PROJECT_ENDPOINT}/agents/{agent_name}?api-version=v1",
         "--resource", "https://ai.azure.com",
         "--query", "instance_identity.principal_id", "-o", "tsv"],
        capture_output=True, text=True, shell=(os.name == "nt"),
    )
    return out.stdout.strip()

for agent_name in (SCOUT_AGENT_NAME, QUILL_AGENT_NAME):
    pid = principal_id(agent_name)
    print(f"\n# {agent_name} identity principal id: {pid or '(not found — check az login / agent name)'}")
    az_args = [
        "role", "assignment", "create",
        "--assignee-object-id", pid,
        "--assignee-principal-type", "ServicePrincipal",
        "--role", ROLE_STORAGE_BLOB_DATA_CONTRIBUTOR,
        "--scope", STORAGE_RESOURCE_ID,
    ]
    if APPLY and pid and "<" not in STORAGE_RESOURCE_ID:
        print(f"Assigning '{ROLE_STORAGE_BLOB_DATA_CONTRIBUTOR}' on {STORAGE_RESOURCE_ID} ...")
        r = subprocess.run(["az", *az_args], capture_output=True, text=True, shell=(os.name == "nt"))
        print(r.stdout or r.stderr)
    else:
        print("az " + " ".join(f'"{a}"' if " " in a else a for a in az_args))

if not APPLY:
    print("\nSet STORAGE_RESOURCE_ID in .env and APPLY = True to run these, or paste them. "
          "Allow ~5-10 min for data-plane RBAC to propagate.")


# scout-research identity principal id: 39a42b40-8fdd-4276-bed3-9d574536d42a
az role assignment create --assignee-object-id 39a42b40-8fdd-4276-bed3-9d574536d42a --assignee-principal-type ServicePrincipal --role "Storage Blob Data Contributor" --scope /subscriptions/0cdd6a6b-2203-4ccb-873f-8eb1652bef4e/resourceGroups/achauhan-pa-3/providers/Microsoft.Storage/storageAccounts/auditdatademo

# quill-analyst identity principal id: b0664ae9-e173-472a-9bb9-4a3e9e6a07a7
az role assignment create --assignee-object-id b0664ae9-e173-472a-9bb9-4a3e9e6a07a7 --assignee-principal-type ServicePrincipal --role "Storage Blob Data Contributor" --scope /subscriptions/0cdd6a6b-2203-4ccb-873f-8eb1652bef4e/resourceGroups/achauhan-pa-3/providers/Microsoft.Storage/storageAccounts/auditdatademo

Set STORAGE_RESOURCE_ID in .env and APPLY = True to run these, or paste them. Allow ~5-10 min for data-plane RBAC to propagate.


## Step 4 — Run the story

Each session is its **own conversation** (its own warm Hand). Watch the `x-agent-session-id`:
it changes between sessions, yet each new session sees the prior work **because it re-mounts the
same blob**. The harness runs asynchronously, so we stream events and poll to a terminal state.

In [4]:
import time

openai_client = project_client.get_openai_client()

def run_turn(agent_name, conversation_id, prompt):
    '''Send one turn to `agent_name` in `conversation_id`; stream, then poll to terminal.'''
    all_events, response_id, terminal, start = [], None, False, time.time()
    with openai_client.responses.with_streaming_response.create(
        conversation=conversation_id,
        model=MODEL,
        input=prompt,
        stream=True,
        extra_body={"agent_reference": {"type": "agent_reference", "name": agent_name}},
    ) as api_response:
        print(f"[{agent_name}] x-agent-session-id: {api_response.headers.get('x-agent-session-id')}")
        print("-" * 80)
        for event in api_response.parse():
            all_events.append(event.type)
            if event.type == "response.created":
                response_id = event.response.id
            elif event.type == "response.output_text.delta":
                print(event.delta, end="", flush=True)
            elif event.type == "response.output_item.added":
                item = getattr(event, "item", None)
                if item is not None and item.type == "function_call":
                    print(f"\n\U0001f527 Tool call: {item.name}")
            elif event.type == "response.output_item.done":
                item = getattr(event, "item", None)
                if item is not None and item.type == "function_call_output":
                    print(f"\n    \U0001f4cb tool output: {getattr(item, 'output', '')[:200]}")
            elif event.type == "response.completed":
                terminal = True
            elif event.type in ("response.failed", "response.incomplete"):
                terminal = True
                print(f"\n\u26a0\ufe0f {event.type}: {getattr(getattr(event, 'response', None), 'error', None)}")
    if response_id and not terminal:
        print("\n\u23f3 stream closed early; polling ...")
        deadline = time.time() + 600
        while time.time() < deadline:
            resp = openai_client.responses.retrieve(response_id)
            if resp.status in ("completed", "failed", "incomplete", "cancelled"):
                print(f"\n[{resp.status}]\n{getattr(resp, 'output_text', '') or ''}")
                break
            time.sleep(3)
    print(f"\n{'=' * 80}\n[{agent_name}] turn done in {time.time() - start:.1f}s\n")

def new_session(label):
    conv = openai_client.conversations.create()
    print(f"\n{'#' * 80}\n# {label}  (conversation {conv.id})\n{'#' * 80}")
    return conv.id

In [5]:
# ── Session 1 — Scout (research) ─────────────────────────────────────────────
s1 = new_session("SESSION 1 · Scout · research")

run_turn(SCOUT_AGENT_NAME, s1,
    "Mount our project drive, then start the research. I'm analyzing the AI meeting-notes "
    "market. The source material is already on the drive under `raw-inbox/` (raw-inbox/sources.csv "
    "lists the companies; raw-inbox/raw/<company>.md has the details). For each company, write a "
    "structured profile to ai-notes-analysis/findings/<company>.md capturing positioning / target "
    "customer, pricing, and key features. Maintain ai-notes-analysis/sources.csv and write "
    "ai-notes-analysis/status.md. Flush before you finish.")

run_turn(SCOUT_AGENT_NAME, s1,
    "Now deepen it: from the same raw-inbox notes, add a 'User sentiment' section (praise + "
    "complaints) and a 'Funding / company size' section to each profile you already wrote. Append "
    "to the existing files — don't start over. Update status.md to 'ready for analysis' and flush.")


################################################################################
# SESSION 1 · Scout · research  (conversation conv_c3e4ba91beaf2cdd00UvAqtiY1mWGwy0BuSWXa2lsuANCChsxE)
################################################################################
[scout-research] x-agent-session-id: cchain_c3e4ba91beaf2cdd00RtPNlXstajpZGAQfnlAFk6wlcqFSdkwC
--------------------------------------------------------------------------------
Done — I mounted the project drive, profiled the 5 companies, updated:

- `ai-notes-analysis/findings/*.md`
- `ai-notes-analysis/sources.csv`
- `ai-notes-analysis/raw_notes.md`
- `ai-notes-analysis/status.md`

Then I flushed the workspace so the files are durable.
[scout-research] turn done in 64.7s

[scout-research] x-agent-session-id: cchain_c3e4ba91beaf2cdd00RtPNlXstajpZGAQfnlAFk6wlcqFSdkwC
--------------------------------------------------------------------------------
Done — I appended **User sentiment** and **Funding / company size** sections to 

In [6]:
# ── Session 2 — Quill (analysis), a BRAND-NEW conversation / Hand ─────────────
s2 = new_session("SESSION 2 · Quill · draft + refine  (new Hand — sees Scout's work via the blob)")

run_turn(QUILL_AGENT_NAME, s2,
    "Mount our project drive and read what the researcher left (status.md, sources.csv, and every "
    "file in ai-notes-analysis/findings/). Draft ai-notes-analysis/report.md with: an executive "
    "summary, a comparison table (company x positioning / pricing / key feature / sentiment), and a "
    "recommendation on where the market has gaps. Update status.md and flush.")

run_turn(QUILL_AGENT_NAME, s2,
    "Refine report.md in place: tighten the executive summary to 5 bullets, and add a "
    "pricing-vs-features quadrant grouping the companies (budget vs premium x thin vs rich "
    "features). Update status.md to 'report v2 ready for review' and flush.")


################################################################################
# SESSION 2 · Quill · draft + refine  (new Hand — sees Scout's work via the blob)  (conversation conv_ae7837147084609200bxm1lyGkncTyzMfysWj464WRiejYNQOR)
################################################################################
[quill-analyst] x-agent-session-id: cchain_ae78371470846092008HPejH4pv9VWGnQGSkorjgS0Rvz0eeEK
--------------------------------------------------------------------------------
Done — I mounted the project drive, read `status.md`, `sources.csv`, and all files in `ai-notes-analysis/findings/`, drafted `ai-notes-analysis/report.md`, updated `status.md`, and flushed the workspace.
[quill-analyst] turn done in 103.4s

[quill-analyst] x-agent-session-id: cchain_ae78371470846092008HPejH4pv9VWGnQGSkorjgS0Rvz0eeEK
--------------------------------------------------------------------------------
Done — `report.md` was refined in place with a tighter 5-bullet executive summary and a pric

In [7]:
# ── Session 3 — a week later: a new competitor appears ───────────────────────
s3a = new_session("SESSION 3a · Scout · profile the new competitor")
run_turn(SCOUT_AGENT_NAME, s3a,
    "Mount our project drive. A new competitor just launched: ClaroNotes "
    "(raw-inbox/late-arrival/claronotes.md). Profile it exactly like the others into "
    "ai-notes-analysis/findings/claronotes.md, add a row to ai-notes-analysis/sources.csv, and note "
    "in status.md that a new competitor was added. Flush before finishing.")

s3b = new_session("SESSION 3b · Quill · fold the newcomer into the report")
run_turn(QUILL_AGENT_NAME, s3b,
    "Mount our project drive and re-read ai-notes-analysis/findings/ — a new competitor (ClaroNotes) "
    "was added. Fold it into report.md: refresh the comparison table, the pricing-vs-features "
    "quadrant, and the recommendation. Refresh, don't rewrite. Update status.md and flush.")


################################################################################
# SESSION 3a · Scout · profile the new competitor  (conversation conv_6c9649705a04d5c200cZzoBptIe5Vp5h371JZLQ2NuDjp3SE0Y)
################################################################################
[scout-research] x-agent-session-id: cchain_6c9649705a04d5c200v9fM3v1SoF6hrNBgQVlPHP3FdTLndxbH
--------------------------------------------------------------------------------
Done. Mounted the drive, profiled ClaroNotes in `ai-notes-analysis/findings/claronotes.md`, added it to `ai-notes-analysis/sources.csv`, updated `status.md`, and flushed the workspace.
[scout-research] turn done in 49.1s


################################################################################
# SESSION 3b · Quill · fold the newcomer into the report  (conversation conv_edb0d6d97df9dd4100Z5eAh3n8OrjU3g5uaQ6YKBlqadVbkX1J)
################################################################################
[quill-analyst] x-agent-s

## Step 5 — Inspect the shared result

The living deliverable is on the blob, not in any one Hand. Ask a fresh session to print it —
demonstrating once more that any new Hand can mount and read the accumulated work.

In [8]:
inspect = new_session("INSPECT · print status.md + report.md from the drive")
run_turn(QUILL_AGENT_NAME, inspect,
    "Mount our project drive and print the full contents of ai-notes-analysis/status.md and then "
    "ai-notes-analysis/report.md. Do not modify anything.")


################################################################################
# INSPECT · print status.md + report.md from the drive  (conversation conv_ddb7093d2c30869e00uAR0B4sJFinif079BkmGLTMgh5oUcw0P)
################################################################################
[quill-analyst] x-agent-session-id: cchain_ddb7093d2c30869e00Aic1momXXLWqKqsyHBzIEt0plXN0kEfC
--------------------------------------------------------------------------------
Mounted the project drive and printed the requested files. No modifications made.

===== ai-notes-analysis/status.md =====
```md
# Status
Owner of last update: Quill
Done: refreshed report.md to fold in ClaroNotes across the comparison table, pricing-vs-features quadrant, recommendations, and supporting summary language.
Next: ready for review. If Scout adds another competitor or materially revises pricing/features, re-read findings/ and refresh the same report in place.
Open questions: ClaroNotes still has no website in sources.

## Step 6 — Clean up

Delete both agent versions. The published skill, the toolbox, the connection, and everything on
the **project drive** (your blob container) remain — that data is the durable artifact.

In [ ]:
for agent in (scout, quill):
    for v in project_client.agents.list_versions(agent_name=agent.name):
        project_client.agents.delete_version(agent_name=agent.name, agent_version=v.version)
    print(f"Deleted versions of {agent.name}")
project_client.close()

## How it works (recap)

- **Two specialists, one drive.** Scout writes `findings/`; Quill writes `report.md`. Neither
  edits the other's files; they coordinate entirely through the shared blob container.
- **Persistence across sessions & agents.** Each conversation is a fresh Hand with empty local
  disk. The blob mount is what carries the work across — a new agent in a new chat mounts the
  same container and continues, with nothing re-explained.
- **Identity, not secrets.** The `workspace-mount` skill mounts via the agent's **managed
  identity** (MSI); no account key or SAS ever enters the model's context. The only grant needed
  is **Storage Blob Data Contributor** on the container.
- **Durability is explicit.** blobfuse2 caches writes locally and uploads on flush/unmount, so
  each agent runs the skill's `flush` before finishing — that is what makes its work visible to
  the next session.
- **Interim pattern.** The in-Hand blobfuse2 mount is a stopgap until native blob-volume
  mounting is available to hosted agents.